In [2]:
from typing import Dict, Any

CITY_ROLE_SCHEDULE_RULES = {
    "wien": {
        "reinigungskraft": [
            (20, 5, 4.0),
            (25, 5, 5.0),
            (30, 5, 6.0),
            (32, 5, 6.4),
            (40, 5, 8.0),
        ],
        "zimmermaedchen": [
            (20, 5, 4.0),
            (40, 5, 8.0),
        ],
    }
}

def validate_contract_selection(attrs: Dict[str, Any]) -> None:
    city = attrs.get("city_code")
    occupation = attrs.get("occupation_code")
    weekly_hours = attrs.get("weekly_hours")
    days_per_week = attrs.get("days_per_week")
    daily_hours = attrs.get("daily_hours")

    if not city:
        raise ValueError("work_city is required to resolve a contract template")

    if not occupation:
        raise ValueError("occupation is required to resolve a contract template")

    city_rules = CITY_ROLE_SCHEDULE_RULES.get(city)
    if not city_rules:
        return  # no special validation for this city

    allowed_combinations = city_rules.get(occupation)
    if not allowed_combinations:
        raise ValueError(
            f"No valid contract rules defined for city '{city}' and occupation '{occupation}'"
        )

    if weekly_hours is None:
        available_hours = sorted({combo[0] for combo in allowed_combinations})
        raise ValueError(
            f"weekly_hours is required for city '{city}' and occupation '{occupation}'. "
            f"Available hours: {', '.join(str(h) for h in available_hours)}"
        )

    if days_per_week is None:
        available_days = sorted({combo[1] for combo in allowed_combinations})
        raise ValueError(
            f"days_per_week is required for city '{city}' and occupation '{occupation}'. "
            f"Available days/week: {', '.join(str(d) for d in available_days)}"
        )

    if daily_hours is None:
        available_daily = sorted({combo[2] for combo in allowed_combinations})
        raise ValueError(
            f"daily_hours is required for city '{city}' and occupation '{occupation}'. "
            f"Available daily hours: {', '.join(str(d) for d in available_daily)}"
        )

    selected = (weekly_hours, days_per_week, daily_hours)
    if selected not in allowed_combinations:
        raise ValueError(
            f"Invalid schedule {selected} for city '{city}' and occupation '{occupation}'. "
            f"Allowed combinations: {allowed_combinations}"
        )

In [3]:
def run_test(attrs, label="Test"):
    try:
        validate_contract_selection(attrs)
        print(f"{label}: PASS")
    except Exception as e:
        print(f"{label}: FAIL -> {e}")

In [4]:
# 1. valid case
run_test({
    "city_code": "wien",
    "occupation_code": "reinigungskraft",
    "weekly_hours": 20,
    "days_per_week": 5,
    "daily_hours": 4.0
}, "Valid Wien Reinigungskraft")

Valid Wien Reinigungskraft: PASS


In [5]:
# 2. missing city
run_test({
    "occupation_code": "reinigungskraft",
    "weekly_hours": 20,
    "days_per_week": 5,
    "daily_hours": 4.0
}, "Missing city")

Missing city: FAIL -> work_city is required to resolve a contract template


In [6]:
# 3. missing occupation
run_test({
    "city_code": "wien",
    "weekly_hours": 20,
    "days_per_week": 5,
    "daily_hours": 4.0
}, "Missing occupation")

Missing occupation: FAIL -> occupation is required to resolve a contract template


In [7]:
# 4. missing weekly_hours
run_test({
    "city_code": "wien",
    "occupation_code": "reinigungskraft",
    "days_per_week": 5,
    "daily_hours": 4.0
}, "Missing weekly hours")

Missing weekly hours: FAIL -> weekly_hours is required for city 'wien' and occupation 'reinigungskraft'. Available hours: 20, 25, 30, 32, 40


In [8]:
# 5. missing days_per_week
run_test({
    "city_code": "wien",
    "occupation_code": "reinigungskraft",
    "weekly_hours": 20,
    "daily_hours": 4.0
}, "Missing days per week")

Missing days per week: FAIL -> days_per_week is required for city 'wien' and occupation 'reinigungskraft'. Available days/week: 5


In [ ]:
# 6. missing daily_hours
run_test({
    "city_code": "wien",
    "occupation_code": "reinigungskraft",
    "weekly_hours": 20,
    "days_per_week": 5
}, "Missing daily hours")

In [9]:
# 7. invalid combination
run_test({
    "city_code": "wien",
    "occupation_code": "reinigungskraft",
    "weekly_hours": 20,
    "days_per_week": 4,
    "daily_hours": 5.0
}, "Invalid combination")

Invalid combination: FAIL -> Invalid schedule (20, 4, 5.0) for city 'wien' and occupation 'reinigungskraft'. Allowed combinations: [(20, 5, 4.0), (25, 5, 5.0), (30, 5, 6.0), (32, 5, 6.4), (40, 5, 8.0)]


In [10]:
# 8. city with no special rules should pass
run_test({
    "city_code": "berlin",
    "occupation_code": "reinigungskraft",
    "weekly_hours": 20,
    "days_per_week": 5,
    "daily_hours": 4.0
}, "Berlin no special rules")


Berlin no special rules: PASS


In [11]:
attrs = {
    "city_code": "wien",
    "occupation_code": "reinigungskraft",
    "weekly_hours": 20,
    "days_per_week": 5,
    "daily_hours": 4.0
}

city = attrs.get("city_code")
occupation = attrs.get("occupation_code")
weekly_hours = attrs.get("weekly_hours")
days_per_week = attrs.get("days_per_week")
daily_hours = attrs.get("daily_hours")

print(city, occupation, weekly_hours, days_per_week, daily_hours)

city_rules = CITY_ROLE_SCHEDULE_RULES.get(city)
print("city_rules:", city_rules)

allowed_combinations = city_rules.get(occupation)
print("allowed_combinations:", allowed_combinations)

print("selected tuple:", (weekly_hours, days_per_week, daily_hours))
print("is valid:", (weekly_hours, days_per_week, daily_hours) in allowed_combinations)


wien reinigungskraft 20 5 4.0
city_rules: {'reinigungskraft': [(20, 5, 4.0), (25, 5, 5.0), (30, 5, 6.0), (32, 5, 6.4), (40, 5, 8.0)], 'zimmermaedchen': [(20, 5, 4.0), (40, 5, 8.0)]}
allowed_combinations: [(20, 5, 4.0), (25, 5, 5.0), (30, 5, 6.0), (32, 5, 6.4), (40, 5, 8.0)]
selected tuple: (20, 5, 4.0)
is valid: True
